# CaVFish Pipeline – Stage 2: Inference

## Automated Keypoint Detection and Morphometric Measurement Extraction

This notebook demonstrates the complete inference workflow for extracting phenotypic measurements from cave fish images using trained deep learning models.

In [ ]:
from pathlib import Path
import os, sys

def find_root(start: Path, marker: str = "custom_transforms.py") -> Path:
    p = start.resolve()
    for candidate in [p, *p.parents]:
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(f"Could not find {marker} above {start}")

PROJECT_ROOT = find_root(Path.cwd(), "custom_transforms.py")

# Make it available to *this* notebook kernel
sys.path.insert(0, str(PROJECT_ROOT))

# Make it available to subprocesses launched from this notebook (e.g., !python ...)
os.environ["PYTHONPATH"] = f"{PROJECT_ROOT}:{os.environ.get('PYTHONPATH','')}"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("PYTHONPATH   =", os.environ["PYTHONPATH"])

---

## Overview

This notebook demonstrates the complete inference workflow for the CaVFish pipeline:

1. **Single Image Inference** - Process individual images and generate keypoint predictions
2. **Batch Inference** - Process multiple images from dataset folders
3. **Phenotypic Measurements** - Compute morphometric measurements from keypoints

All examples use the sample data provided in `demo/files/` for reproducibility.

In [ ]:
## Download pre-trained weights (optional)
# Run this cell to download the trained models from GitHub Releases

import subprocess
import os
from pathlib import Path

# Create checkpoints directory if it doesn't exist
checkpoint_dir = PROJECT_ROOT / 'checkpoints'
checkpoint_dir.mkdir(exist_ok=True)

# Model URLs
models = {
    'baseline.pth': 'https://github.com/josuedelarosa/CaVFish-MORPH/releases/download/baseline/baseline.pth',
    'baseline-log.pth': 'https://github.com/josuedelarosa/CaVFish-MORPH/releases/download/baseline-log/baseline-log.pth',
    'phenoloss.pth': 'https://github.com/josuedelarosa/CaVFish-MORPH/releases/download/phenoloss/phenoloss.pth',
    'phenoloss-log.pth': 'https://github.com/josuedelarosa/CaVFish-MORPH/releases/download/phenoloss-log/phenoloss-log.pth',
}

# Download each model
for model_name, url in models.items():
    model_path = checkpoint_dir / model_name
    
    if model_path.exists():
        print(f"✓ {model_name} already exists")
    else:
        print(f"Downloading {model_name}...")
        try:
            subprocess.run(['wget', '-O', str(model_path), url], check=True)
            print(f"✓ Downloaded {model_name}")
        except subprocess.CalledProcessError as e:
            print(f"✗ Failed to download {model_name}: {e}")
        except FileNotFoundError:
            print(f"wget not found. Try using curl instead or download manually from {url}")

print(f"\n✅ Models available in: {checkpoint_dir}")

---

## 📥 Download Pre-trained Weights (Optional)

Pre-trained model weights for all 4 experiments are available in the [GitHub Releases](https://github.com/josuedelarosa/CaVFish-MORPH/releases).

**Available Models:**
| Experiment | URL |
|-----------|-----|
| **Baseline (MSE)** | https://github.com/josuedelarosa/CaVFish-MORPH/releases/download/baseline/baseline.pth |
| **Baseline+Log** | https://github.com/josuedelarosa/CaVFish-MORPH/releases/download/baseline-log/baseline-log.pth |
| **PhenoLoss** | https://github.com/josuedelarosa/CaVFish-MORPH/releases/download/phenoloss/phenoloss.pth |
| **PhenoLoss+Log** | https://github.com/josuedelarosa/CaVFish-MORPH/releases/download/phenoloss-log/phenoloss-log.pth |

You can skip training and directly use these pre-trained weights for inference.

In [ ]:
## Single Image Inference
# Process one image and generate visualization + JSON keypoint predictions

# Using pre-trained weights (on GitHub Release)
!python ../demo/image_demo.py \
    ../demo/files/cavfish1.jpg \
    ../configs/experiment1_baseline_mse.py \
    ../checkpoints/baseline.pth \
    --out-file ../demo/files/result.jpg \
    --radius 5 \
    --kpt-thr 0.3 \
    --device cpu

# Output:
# - result.jpg: Visualization with keypoints overlaid
# - result_keypoints.json: JSON with keypoint coordinates and confidence scores

In [ ]:
## Batch Inference
# Process multiple images and create consolidated JSON output
# Using pre-trained baseline model weights

!python ../demo/cavfish_batch_inference.py \
    --config ../configs/experiment1_baseline_mse.py \
    --checkpoint ../checkpoints/baseline.pth \
    --model baseline_mse \
    --dataset-root ../demo/ \
    --datasets "files" \
    --device cpu

# Output:
# - Creates: ../demo/baseline_mse/inference_files/all_keypoints_predicted_files.json
# - Contains: Consolidated keypoints for all images in the dataset

---

## Command-Line Options

### `image_demo.py` - Single Image Inference

**Basic Usage:**
```bash
python demo/image_demo.py <image> <config> <checkpoint> [options]
```

**Options:**
- `--out-file` - Output visualization path (default: result.jpg)
- `--radius` - Keypoint visualization radius (default: 3)
- `--kpt-thr` - Confidence threshold for displaying keypoints (default: 0.3)
- `--draw-heatmap` - Include heatmap visualization
- `--show-kpt-idx` - Display keypoint indices
- `--device` - Compute device: `cpu`, `cuda:0`, `cuda:1`, etc. (default: cuda:0)

### `cavfish_batch_inference.py` - Batch Processing

**Basic Usage:**
```bash
python demo/cavfish_batch_inference.py --config <config> --checkpoint <checkpoint> --model <name> [options]
```

**Options:**
- `--model` - Model identifier for output folder naming (required)
- `--dataset-root` - Root directory containing dataset folders
- `--datasets` - Specific dataset folders to process (space-separated)
- `--with-heatmap` - Request heatmap outputs from model
- `--device` - Compute device: `cpu`, `cuda:0`, `cuda:1`, etc. (default: cuda:0)

**Note:** Results are cached - existing JSON files will be reused unless corrupted.

---

## Output Formats

### Single Image Output

**Files generated:**
- `<out-file>`: Visualization with keypoints overlaid
- `<out-file>_keypoints.json`: Keypoint predictions

**JSON structure:**
```json
{
  "image": "path/to/image.jpg",
  "keypoints": [
    {"name": "0", "x": 123.4, "y": 456.7, "score": 0.98},
    {"name": "1", "x": 234.5, "y": 567.8, "score": 0.95},
    ...
  ]
}
```

### Batch Inference Output

**Directory structure:**
```
<DATASET_ROOT>/
└── <MODEL_NAME>/
    └── inference_<dataset>/
        ├── all_keypoints_predicted_<dataset>.json  # Consolidated
        └── <subdirs>/
            └── *_keypoints.json  # Individual predictions
```

**Consolidated JSON format:**
```json
[
  {
    "image": "dataset/subdir/fish001.jpg",
    "keypoints": [...]
  },
  {
    "image": "dataset/subdir/fish002.jpg",
    "keypoints": [...]
  }
]
```

In [ ]:
## Compute Morphometric Measurements
# Calculate phenotypic measurements from keypoint predictions

!python ../demo/compute_phenotypic_measurements.py \
    ../demo/files/result_keypoints.json \
    --output ../demo/files/measurements.csv \
    --format csv

# Output CSV contains:
# - Absolute measurements (in pixels): BI, Bd, Hd, CPd, CFd, Ed, Eh, JI, PFI, PFi, HL, DL, AL
# - Normalized measurements (relative to BI): <measurement>_norm for all metrics
# - Mean confidence score for each image

---

## Phenotypic Measurements

### Measurement Definitions

The `compute_phenotypic_measurements.py` script computes 13 standard morphometric measurements from the 20 detected keypoints:

| Measurement | Keypoints | Description |
|------------|-----------|-------------|
| **BI** | 0 → 1 | Body Index: Upper Snout Tip → Caudal Peduncle Center |
| **Bd** | 2 → 3 | Body Depth: Dorsal Body → Pelvic Fin Base |
| **Hd** | 4 → 5 | Head Depth: Upper Head → Barbel Base |
| **CPd** | 6 → 7 | Caudal Peduncle Depth: Mid-Dorsal Trunk → Ventral Trunk |
| **CFd** | 8 → 9 | Caudal Fin Depth: Upper Caudal Base → Lower Caudal Fin Tip |
| **Ed** | 10 → 11 | Eye Diameter: Eye Center → Posterior Eye Margin |
| **Eh** | 12 → 3 | Eye Height: Lower Eye Margin → Pelvic Fin Base |
| **JI** | 0 → 13 | Jaw Index: Upper Snout Tip → Lower Jaw Tip |
| **PFI** | 14 → 15 | Pelvic Fin Index: Operculum Lower Edge → Pelvic Fin Tip |
| **PFi** | 14 → 3 | Pelvic Fin insertion: Operculum Lower Edge → Pelvic Fin Base |
| **HL** | 0 → 16 | Head Length: Upper Snout Tip → Operculum Upper Edge |
| **DL** | 2 → 17 | Dorsal Length: Dorsal Body → Dorsal Fin Tip |
| **AL** | 18 → 19 | Anal Length: Anal Fin Base → Anal Fin Tip |

### Output Format

**CSV/Excel columns:**
- `image` - Source image filename
- `<measurement>` - Absolute measurement in pixels (e.g., SL, HL, DFL)
- `<measurement>` - Absolute measurement in pixels (e.g., BI, Bd, Hd)
- `<measurement>_norm` - Size-normalized measurement (relative to BI)

**Example:** `HL_norm = HL / SL` provides size-independent head proportion.
**Example:** `HL_norm = HL / BI` provides size-independent head proportion.
**Formats supported:** CSV, Excel (.xlsx), JSON

In [ ]:
## Inspect Measurements
import pandas as pd
from pathlib import Path

# Load the measurements
measurements_file = PROJECT_ROOT / 'demo' / 'files' / 'measurements.csv'

if measurements_file.exists():
    df = pd.read_csv(measurements_file)
    
    # Display summary
    print(f"Total fish analyzed: {len(df)}")
    print(f"\n{'='*80}")
    print("SAMPLE MEASUREMENTS")
    print('='*80)
    print(df.head())
    
    # Show key statistics
    print(f"\n{'='*80}")
    print("SUMMARY STATISTICS")
    print('='*80)
    
    # Select key measurements for display
    key_metrics = ['SL', 'HL', 'HL_norm', 'mean_confidence']
    available_metrics = [m for m in key_metrics if m in df.columns]
    
    if available_metrics:
        print(df[available_metrics].describe())
    else:
        print("Note: Run the measurement computation cell above to generate statistics")
else:
    print(f"Measurements file not found: {measurements_file}")
    print("Please run the 'Compute Morphometric Measurements' cell above first.")

---

## Next Steps

After completing inference and measurement extraction, proceed to:

**Stage 3: Morphometric Analysis** ([3-measurements.ipynb](3-measurements.ipynb))
- Statistical analysis of phenotypic measurements
- Visualization of morphometric patterns
- Comparative analysis across specimens

---